# Phase 6: Advanced Analysis

The forecast models are evaluated on a six-month holdout period before future predictions are produced.

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression

project_folder = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed_folder = project_folder / 'data' / 'processed'
sales = pd.read_csv(processed_folder / 'cleaned_sales.csv', parse_dates=['Order Date'])
print('Rows loaded:', len(sales))

## 1. Prepare complete monthly sales

The source ends on 20 February 2021. February is incomplete, so it is excluded from model training and forecast evaluation.

In [ ]:
monthly_sales = (
    sales.set_index('Order Date')['Revenue USD']
    .resample('MS')
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'Month', 'Revenue USD': 'Actual Revenue'})
)

final_observed_month = sales['Order Date'].max().to_period('M').to_timestamp()
complete_monthly_sales = monthly_sales[monthly_sales['Month'] < final_observed_month].copy()
complete_monthly_sales['Month Number'] = range(len(complete_monthly_sales))

print('Last complete month:', complete_monthly_sales['Month'].max().date())
display(complete_monthly_sales.tail())

## 2. Define the two forecast methods

In [ ]:
def moving_average_forecast(history, periods):
    values = list(history)
    predictions = []
    for _ in range(periods):
        prediction = sum(values[-3:]) / 3
        predictions.append(prediction)
        values.append(prediction)
    return predictions


def linear_regression_forecast(history, periods):
    training_data = pd.DataFrame({
        'Month Number': range(len(history)),
        'Revenue': list(history),
    })
    future_data = pd.DataFrame({
        'Month Number': range(len(history), len(history) + periods)
    })
    model = LinearRegression()
    model.fit(training_data[['Month Number']], training_data['Revenue'])
    return model.predict(future_data)

## 3. Evaluate both models on the last six complete months

In [ ]:
training_revenue = complete_monthly_sales['Actual Revenue'].iloc[:-6]
test_revenue = complete_monthly_sales['Actual Revenue'].iloc[-6:].reset_index(drop=True)
moving_test = pd.Series(moving_average_forecast(training_revenue, 6))
regression_test = pd.Series(linear_regression_forecast(training_revenue, 6))

def calculate_errors(actual, predicted):
    absolute_error = (actual - predicted).abs()
    mae = absolute_error.mean()
    mape = (absolute_error / actual).mean() * 100
    return round(mae, 2), round(mape, 2)

moving_mae, moving_mape = calculate_errors(test_revenue, moving_test)
regression_mae, regression_mape = calculate_errors(test_revenue, regression_test)

forecast_validation = pd.DataFrame({
    'Model': ['Three-Month Moving Average', 'Linear Regression'],
    'MAE': [moving_mae, regression_mae],
    'MAPE %': [moving_mape, regression_mape],
}).sort_values('MAE')
selected_model = forecast_validation.iloc[0]['Model']
forecast_validation.to_csv(processed_folder / 'forecast_validation.csv', index=False)
display(forecast_validation)
print('Selected model:', selected_model)

## 4. Forecast the next three and six months

In [ ]:
all_complete_revenue = complete_monthly_sales['Actual Revenue']
moving_future = moving_average_forecast(all_complete_revenue, 6)
regression_future = linear_regression_forecast(all_complete_revenue, 6)
future_months = pd.date_range(
    start=complete_monthly_sales['Month'].max() + pd.offsets.MonthBegin(1),
    periods=6,
    freq='MS',
)
sales_forecast = pd.DataFrame({
    'Month': future_months,
    'Moving Average Forecast': moving_future,
    'Linear Regression Forecast': regression_future,
    'Horizon Month': range(1, 7),
    'Included in 3-Month Forecast': ['Yes'] * 3 + ['No'] * 3,
})
sales_forecast[['Moving Average Forecast', 'Linear Regression Forecast']] = (
    sales_forecast[['Moving Average Forecast', 'Linear Regression Forecast']].round(2)
)
sales_forecast.to_csv(processed_folder / 'sales_forecast.csv', index=False, date_format='%Y-%m-%d')
display(sales_forecast)

## 5. Market-basket analysis

A pair is retained only when it appears in at least three orders, has support of at least 0.01%, confidence of at least 1% in either direction, and lift above 1. These limits remove one-off coincidences while retaining enough pairs for review.

In [ ]:
products_by_order = sales.groupby('Order Number')['Product Name'].apply(lambda values: sorted(set(values)))
product_order_counts = Counter()
pair_order_counts = Counter()

for products in products_by_order:
    product_order_counts.update(products)
    pair_order_counts.update(combinations(products, 2))

total_orders = len(products_by_order)
basket_rows = []
for (product_a, product_b), pair_count in pair_order_counts.items():
    support = pair_count / total_orders
    confidence_a_to_b = pair_count / product_order_counts[product_a]
    confidence_b_to_a = pair_count / product_order_counts[product_b]
    lift = support / (
        (product_order_counts[product_a] / total_orders)
        * (product_order_counts[product_b] / total_orders)
    )
    if pair_count >= 3 and support >= 0.0001 and max(confidence_a_to_b, confidence_b_to_a) >= 0.01 and lift > 1:
        basket_rows.append({
            'Product A': product_a,
            'Product B': product_b,
            'Orders Together': pair_count,
            'Support %': round(support * 100, 4),
            'Confidence A to B %': round(confidence_a_to_b * 100, 2),
            'Confidence B to A %': round(confidence_b_to_a * 100, 2),
            'Lift': round(lift, 2),
        })

market_basket = pd.DataFrame(basket_rows).sort_values(['Orders Together', 'Lift'], ascending=False)
market_basket.to_csv(processed_folder / 'market_basket.csv', index=False)
print('Pairs meeting the thresholds:', len(market_basket))
display(market_basket.head(20))

## 6. Product profitability groups

High and low are relative to the median revenue, gross profit, and gross margin of products in this dataset. They are exploratory groups, not company targets.

In [ ]:
product_profitability = (
    sales.groupby(['ProductKey', 'Product Name'], as_index=False)
    .agg(Revenue=('Revenue USD', 'sum'), Gross_Profit=('Gross Profit USD', 'sum'))
)
product_profitability['Gross Margin %'] = (
    product_profitability['Gross_Profit'] / product_profitability['Revenue'] * 100
)
median_revenue = product_profitability['Revenue'].median()
median_profit = product_profitability['Gross_Profit'].median()
median_margin = product_profitability['Gross Margin %'].median()

def assign_profitability_group(row):
    if row['Revenue'] >= median_revenue and row['Gross Margin %'] < median_margin:
        return 'High Sales Low Profit'
    if row['Revenue'] < median_revenue and row['Gross_Profit'] >= median_profit:
        return 'High Profit Low Sales'
    return 'Other Products'

product_profitability['Profitability Group'] = product_profitability.apply(assign_profitability_group, axis=1)
product_profitability[['Revenue', 'Gross_Profit', 'Gross Margin %']] = (
    product_profitability[['Revenue', 'Gross_Profit', 'Gross Margin %']].round(2)
)
product_profitability.to_csv(processed_folder / 'product_profitability.csv', index=False)
display(product_profitability['Profitability Group'].value_counts())